In [ ]:
# kütüphane + drive bağlantı
!pip install transformers==4.44.0 torch datasets accelerate -q

from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/deutschify_t5/'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Kayıt klasörü hazır: {SAVE_DIR}')

In [ ]:
# dataset + tokenizasyon
import json
import urllib.request
import torch
from torch.utils.data import Dataset
from transformers import T5Tokenizer

DATASET_URL = (
    'https://raw.githubusercontent.com/zehranuracikgoz/Deutschify'
    '/main/dataset/t5_train.json'
)

print('Dataset indiriliyor')
with urllib.request.urlopen(DATASET_URL) as resp:
    raw_data = json.loads(resp.read().decode('utf-8'))
print(f'Yüklenen kayıt sayısı: {len(raw_data)}')
print(f'Örnek -> input : {raw_data[0]["input"]}')
print(f'output: {raw_data[0]["output"]}')

print('\nTokenizer yükleniyor')
tokenizer = T5Tokenizer.from_pretrained('t5-small')

MAX_INPUT_LEN  = 32
MAX_TARGET_LEN = 64

class DeutschifyDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        input_enc = self.tokenizer(
            item['input'],
            max_length=MAX_INPUT_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        target_enc = self.tokenizer(
            item['output'],
            max_length=MAX_TARGET_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )

        labels = target_enc['input_ids'].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids':      input_enc['input_ids'].squeeze(),
            'attention_mask': input_enc['attention_mask'].squeeze(),
            'labels':         labels,
        }

dataset = DeutschifyDataset(raw_data, tokenizer)
print(f'\nDataset hazır — {len(dataset)} kayıt, {MAX_INPUT_LEN} / {MAX_TARGET_LEN} token limiti')

In [ ]:
# eğitim manuel pytorch döngüsü
from transformers import T5ForConditionalGeneration
from torch.utils.data import DataLoader
import torch

model = T5ForConditionalGeneration.from_pretrained('t5-small')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

for epoch in range(3):
    model.train()
    total_loss = 0
    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids,
                       attention_mask=attention_mask,
                       labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f'Epoch {epoch+1}/3 — Loss: {avg_loss:.4f}')

    epoch_dir = SAVE_DIR + f'epoch_{epoch+1}/'
    model.save_pretrained(epoch_dir)
    tokenizer.save_pretrained(epoch_dir)
    print(f'Model kaydedildi: {epoch_dir}')

print('Eğitim tamamlandı')

In [ ]:
# test
from transformers import T5ForConditionalGeneration, T5Tokenizer

print('Kaydedilen model yükleniyor')
model     = T5ForConditionalGeneration.from_pretrained(SAVE_DIR + 'epoch_3/')
tokenizer = T5Tokenizer.from_pretrained(SAVE_DIR + 'epoch_3/')
model.eval()

def ornek_uret(kelime: str) -> str:
    prompt = f'örnek_üret: {kelime}'
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        max_length=32,
        truncation=True,
    )
    with torch.no_grad():
        outputs = model.generate(
            inputs['input_ids'],
            max_new_tokens=64,
            num_beams=4,
            no_repeat_ngram_size=2,
            early_stopping=True,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_words = ['Abend', 'Wasser', 'Haus', 'Kind', 'lernen']
print('\n--- Test Sonuçları ---')
for word in test_words:
    result = ornek_uret(word)
    print(f'örnek_üret: {word}')
    print(f'->{result}\n')

In [ ]:
# bleu skoru
!pip install sacrebleu -q

import random
from sacrebleu.metrics import BLEU

random.seed(42)
sample = random.sample(raw_data, min(50, len(raw_data)))

bleu = BLEU(effective_order=True)
scores = []

print('BLEU Değerlendirmesi (50 örnek):')
for item in sample:
    kelime = item['input'].split(': ', 1)[1]
    pred   = ornek_uret(kelime)
    score  = bleu.sentence_score(pred, [item['output']]).score
    scores.append(score)

avg_bleu = sum(scores) / len(scores)
print(f'Ortalama BLEU skoru (50 örnek): {avg_bleu:.2f}')

In [ ]:
# rouge skoru
!pip install rouge-score -q

from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

r1_scores, r2_scores, rL_scores = [], [], []

print('ROUGE Değerlendirmesi (50 örnek):')
for item in sample:
    kelime = item['input'].split(': ', 1)[1]
    pred   = ornek_uret(kelime)
    ref    = item['output']
    result = scorer.score(ref, pred)
    r1_scores.append(result['rouge1'].fmeasure)
    r2_scores.append(result['rouge2'].fmeasure)
    rL_scores.append(result['rougeL'].fmeasure)

print(f'ROUGE-1 F1 (ort.): {sum(r1_scores) / len(r1_scores):.4f}')
print(f'ROUGE-2 F1 (ort.): {sum(r2_scores) / len(r2_scores):.4f}')
print(f'ROUGE-L F1 (ort.): {sum(rL_scores) / len(rL_scores):.4f}')

In [ ]:
# bertscore
!pip install bert-score -q

from bert_score import score as bert_score

hypotheses = []
references  = []

for item in sample:
    kelime = item['input'].split(': ', 1)[1]
    hypotheses.append(ornek_uret(kelime))
    references.append(item['output'])

print('BERTScore Değerlendirmesi (50 örnek):')
P, R, F1 = bert_score(hypotheses, references, lang='de', verbose=False)

print(f'BERTScore F1 (ort.): {F1.mean().item():.4f}')
print(f'BERTScore P  (ort.): {P.mean().item():.4f}')
print(f'BERTScore R  (ort.): {R.mean().item():.4f}')